# Advanced Kernel Reading Guide: DeepGEMM and FlashAttention-3

> Appendix G explained the online softmax + tiling pair behind FlashAttention, answering the question "why does tiling save memory" at the mathematical level. But to actually drive FP8 GEMM to 70% of H100's theoretical peak, knowing the algorithm is not enough — you also have to read real production-grade kernel source code and see how TMA, WGMMA, and warp specialization are wired together.
>
> This appendix does not rewrite any kernel. It is a reading map: it tells the reader in order which source files to open, what each file is responsible for, and where the key functions live. It covers the three most representative classes of advanced kernels in current industry practice — DeepSeek's FP8 GEMM library DeepGEMM, FlashAttention-2/3, and the Triton implementation of grouped GEMM.


Advanced kernels are "advanced" not because the algorithms are complex, but because they have to balance three constraints simultaneously: the characteristics of hardware instructions (TMA and WGMMA on Hopper are asynchronous), the SRAM capacity ceiling (a single block typically has only 100-200 KB available), and numerical precision (FP8's dynamic range is so narrow that per-block scaling is mandatory). Algorithmic tricks such as online softmax and tiling already have their intuition built in Appendix G; the job of this appendix is to connect that intuition to the variable names, function signatures, and file paths that appear in real code.

Let us agree on some terminology first. **TMA (Tensor Memory Accelerator)** is the hardware DMA unit introduced in Hopper; it can move a tile of a tensor from HBM to shared memory asynchronously without the SM's involvement. **WGMMA (Warp-Group Matrix Multiply Accumulate)** is Hopper's asynchronous tensor core instruction; a single instruction performs a 64×N×16 matrix multiply-accumulate. **Warp specialization** is the programming model introduced in CUTLASS 4.x that splits the warps inside a CTA into producer warps (running DMA, moving data) and consumer warps (running MMA, doing the math), synchronized with `mbarrier`. These three terms will reappear throughout.

This appendix contains no benchmarks. Running an FP8 kernel on H100 requires the combination of the Hopper architecture, the CUTLASS main branch, and CUDA 12.3+, which a teaching environment cannot reliably provide. Our goal is for the reader to be able to open the source, understand it, locate the key code segments, and judge which design corresponds to which sentence in the paper.


## 1. Why Cover Advanced Kernels Separately

The implementations in the earlier sections all used the "teaching-grade" abstraction of PyTorch + Triton. The resulting code runs correctly, but it is an order of magnitude away from production-grade performance. The gap comes down to three things.

First, **PyTorch's eager mode is hardware-unaware**. `torch.matmul(a, b)` on an H100 may dispatch to a cuBLAS FP8 kernel, or it may fall back to a BF16 path, depending on dtype, shape, alignment, and a pile of other conditions. To squeeze the hardware, you either write your own kernel or use a library that can JIT a specialized kernel.

Second, **low precision like FP8 is not "just change the dtype"**. The E4M3 format of FP8 has only 4 exponent bits, giving a dynamic range of roughly $\pm 448$, an order of magnitude smaller than BF16. Casting BF16 weights to FP8 directly will most likely overflow or underflow. The production approach is per-block scaling: cut the matrix into 128×128 blocks, compute a separate scale factor for each block, and squeeze the data into the representable range of FP8. This mechanism is the core of what libraries like DeepGEMM do.

Third, **MoE's grouped GEMM cannot reuse a normal GEMM kernel**. A dense model handles one complete matrix per GEMM; a MoE forward has to run a GEMM for each of $E$ experts, and each expert gets a different number of tokens. Naively looping over the $E$ experts, the kernel launch overhead eats half the time. Grouped GEMM packs the $E$ GEMMs into a single kernel launch and lets one CTA process one tile of one expert — something a dense GEMM kernel cannot do.

These three points are the origin of the three classes of kernels covered in this appendix: DeepGEMM solves FP8 + per-block scaling, FlashAttention-3 solves the overlap of asynchronous instructions, and grouped GEMM solves the MoE batched-small-matrix problem.


## 2. DeepGEMM Source Reading Guide

DeepGEMM is the FP8 GEMM library that DeepSeek open-sourced in February 2025; the repository is [deepseek-ai/DeepGEMM](https://github.com/deepseek-ai/DeepGEMM). Its design scope is intentionally narrow: it only does FP8 dense and grouped GEMM on the Hopper architecture, with no attempt at cross-architecture compatibility. This narrow scope lets it stay minimal — the core is less than 3000 lines of CUDA, whereas the CUTLASS Epilogue alone for the equivalent functionality exceeds 20000 lines.

The two signature designs of DeepGEMM are: a **TMA-friendly layout for per-block scaling factors**, and a **mixed-precision path with BF16 output + FP32 accumulation**. The former keeps the quantization error of FP8 data under control; the latter preserves accumulation accuracy.


### 2.1 The TMA Layout for per-block Scaling

The standard practice for FP8 GEMM is to split the matrix into $B_m \times B_n$ blocks (typically $128 \times 128$) and compute one scale factor $s_{ij}$ per block. During computation, $C_{ij} = \sum_k (A_{ik} \cdot s^A_{ik}) (B_{kj} \cdot s^B_{kj})$, and the output is then multiplied by the corresponding output scale.

The engineering difficulty of this scheme is that the scale factor is itself a tensor and must be loaded into SRAM along with the data. If the layout of the scale does not align with the boundaries of the data blocks, TMA has to do an extra gather when moving data, and performance drops by half.

DeepGEMM's solution is to store the scale factors contiguously along the K dimension and align them strictly with the K tiling of the data blocks. This way a TMA descriptor can describe the entire scale tensor with a single pointer plus two strides, with no complex index arithmetic. This layout is called `NVTE_LAYOUT_BLOCK_SCALING` in the source; it is essentially a three-dimensional tensor $[M / B_m, K / B_k, 1]$ with one scale value per row.

This design has a side benefit: the memory footprint of the scale factor is tiny. For a $4096 \times 4096$ matrix with $B_m = B_k = 128$, the scale tensor has only $32 \times 32 = 1024$ FP32 values — 4 KB — which is essentially negligible. This is exactly the advantage of per-block scaling over per-tensor scaling: an order of magnitude better precision at almost no memory cost.


### 2.2 BF16 Accumulation and the FP8 Precision Balance

The accumulator of an FP8 tensor core is FP32 (this is fixed by hardware and cannot be changed). But the final output usually has to be written back as BF16 — because downstream attention and FFN use BF16 as their base dtype. In between there is a precision choice: after the FP32 accumulation, do we truncate directly to BF16, or do we round?

DeepGEMM chose stochastically rounded BF16 output. The epilogue function in the source calls the stochastic rounding variant of `__nv_cvt_fp32_to_bf16`, so that the statistical expectation of a large number of accumulated results is unbiased. This matters a lot in large-model training — with truncation, the forward error biases in one direction, and over thousands of accumulated layers it can push the model off course.

The accumulation process itself also has subtleties. The accumulator of a WGMMA instruction lives in registers as FP32, but across the multiple WGMMA calls along the K dimension, the accumulated result stays in the same set of registers. If K is long (say $K = 8192$), the accumulated value grows larger and larger, so the later multiply-adds are effectively "big number + small number", and the precision of the small number gets eaten. DeepGEMM periodically "normalizes" the accumulator: it stores the current partial sum into an FP32 "outer accumulator", then zeroes the inner accumulator and starts the next segment from 0. This inner-outer two-level accumulator structure is a common pattern in CUTLASS Hopper GHMA kernels.


### 2.3 Grid and Block Design for Kernel Launch

DeepGEMM's GEMM kernel uses the classic 2D grid: `grid = (M / TM, N / TN)`, where each CTA is responsible for one output tile $C_{ij}$ of size $T_M \times T_N$ (typically $128 \times 256$). Inside the CTA, 4 warps are split into two groups:

- **producer warp group** (1 warp): runs TMA load instructions, moving the next tile of A and B data from HBM into shared memory
- **consumer warp group** (3 warps): runs WGMMA instructions, performing matrix multiply-accumulate on the data in shared memory and writing the result to the register file

The two groups are synchronized with `mbarrier` (Hopper's asynchronous barrier). The producer signals the barrier after moving a tile, and the consumer starts computing when it sees the barrier complete. This producer/consumer split is the core of warp specialization.

The choice of block is the key to performance tuning. $T_M \times T_N \times T_K$ determines SRAM usage (each stage needs $T_M \cdot T_K + T_K \cdot T_N$ FP8 bytes), register usage (the accumulator is $T_M \cdot T_N$ FP32s), and the number of pipeline stages. DeepGEMM uses a 4-stage pipeline by default — while the producer is processing tile $i$, the consumer is computing tile $i-1$, and the producer has already started the TMA load for tile $i+1$. This multi-stage overlap is a necessary condition for driving FP8 GEMM past 70% of peak throughput on Hopper.


### 2.4 Source Reading Route

DeepGEMM is organized much more simply than CUTLASS; reading it in the order below, the entire call chain can be covered in a day or two. All paths are relative to the repository root.

**Entry layer (Python):**

- `deep_gemm/__init__.py` — Python package entry, exports top-level functions such as `fp8_gemm` and `m_grouped_fp8_gemm`
- `deep_gemm/dispatcher.py` — selects the most suitable JIT template based on the (M, N, K) shape and dtype
- `deep_gemm/jit_kernels/gemm.py` — JIT compilation entry, injects shape constants into the CUDA template and invokes NVRTC

**Host-side launch (C++/CUDA):**

- `csrc/deep_gemm.cuh` — the C++ launch entry, sets up grid/block and calls `cudaLaunchKernel`
- `csrc/utils.cuh` — utilities for constructing TMA descriptors, converting a PyTorch tensor into a `CUtensorMap`

**Device kernel (CUDA):**

- `csrc/includes/kernels/gemm/kernel.cuh` — the main kernel template, defines the producer and consumer functions
- `csrc/includes/kernels/gemm/warp_specializer.cuh` — warp specialization strategy, defines mbarrier initialization and wait
- `csrc/includes/kernels/gemm/epilogue.cuh` — the epilogue, FP32 accumulator to BF16 plus applying the per-block scale
- `csrc/includes/utils/matmul.cuh` — wrapper around the WGMMA instruction, packaging PTX inline asm as a C++ function

**Grouped GEMM variant:**

- `csrc/includes/kernels/grouped_gemm/kernel.cuh` — the grouped version used by MoE, where each CTA processes one tile of one expert
- `deep_gemm/jit_kernels/grouped_gemm.py` — JIT entry for the grouped version, handles the stride computation for variable-length experts

Suggested reading order: start with `__init__.py` to see the shape of the public API, then jump to `dispatcher.py` to understand the mapping from shape to kernel, and finally go into `kernel.cuh` to see the loop structure of producer/consumer. `warp_specializer.cuh` and `matmul.cuh` are detail-level and can be read last.


In [ ]:
# Print DeepGEMM's reading route as a checklist you can follow along
deepgemm_reading_path = [
    ("Entry layer (Python)", [
        ("deep_gemm/__init__.py",            "Package entry, exports top-level functions like fp8_gemm"),
        ("deep_gemm/dispatcher.py",          "Picks the JIT template based on (M,N,K) shape"),
        ("deep_gemm/jit_kernels/gemm.py",    "Injects constants into the CUDA template and compiles via NVRTC"),
    ]),
    ("Host launch (C++/CUDA)", [
        ("csrc/deep_gemm.cuh",   "Launch entry, sets grid/block and launches the kernel"),
        ("csrc/utils.cuh",       "Builds CUtensorMap, converts a PyTorch tensor into a TMA descriptor"),
    ]),
    ("Device kernel (CUDA)", [
        ("csrc/includes/kernels/gemm/kernel.cuh",          "Main kernel, the producer/consumer two-segment loop"),
        ("csrc/includes/kernels/gemm/warp_specializer.cuh", "Warp specialization + mbarrier synchronization"),
        ("csrc/includes/kernels/gemm/epilogue.cuh",        "FP32 -> BF16 + applying per-block scale"),
        ("csrc/includes/utils/matmul.cuh",                 "PTX inline asm wrapper for the WGMMA instruction"),
    ]),
    ("Grouped GEMM (MoE variant)", [
        ("csrc/includes/kernels/grouped_gemm/kernel.cuh",  "Each CTA processes one tile of one expert"),
        ("deep_gemm/jit_kernels/grouped_gemm.py",          "Stride computation and JIT for variable-length experts"),
    ]),
]

print("DeepGEMM source reading route")
print("=" * 70)
for layer, files in deepgemm_reading_path:
    print(f"\n[{layer}]")
    for path, desc in files:
        print(f"  {path}")
        print(f"      -> {desc}")
print()
print("Key observation: from __init__.py all the way to kernel.cuh,")
print("        each layer does exactly one thing — pick a template, inject constants, launch, move data, compute.")


## 3. Implementation Details of FlashAttention-2/3

Appendix G covered the FlashAttention algorithm (online softmax + tiling). This appendix focuses on what FA-2 and FA-3 do beyond the algorithm in terms of engineering optimization. The algorithm itself was finalized in FA-1; the improvements in the next two generations concentrate on "how to keep the GPU busier".

The source lives at [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention), and splits into two paths: the CUDA implementation (`csrc/flash_attn/`) and the Triton implementation (`flash_attn/flash_attn_triton.py`). Production uses the CUDA version for performance; the Triton version is for teaching and quick experimentation.


### 3.1 FlashAttention-2: Two Improvements

FA-2 (Dao 2023) relative to FA-1 can be summarized in two sentences: **fewer non-matmul FLOPs, better parallelization**.

The first point comes from a hardware fact: on a GPU the throughput of tensor cores is much higher than that of other operations (on an A100, BF16 matmul is 312 TFLOPs, but element-wise adds and softmax are only tens of TFLOPs). The inner loop of FA-1 has some non-matmul operations (for example, computing $\exp(m_\text{old} - m_\text{new})$ during the rescale); these operations are not dense, but they drag down overall throughput. FA-2 defers the rescale to the outer loop, leaving the inner loop with almost nothing but the two matmuls `Q @ K.T` and `P @ V`. As a result, tensor core utilization climbs from FA-1's roughly 40% to 70%+.

The second point is the change in parallelization strategy. FA-1 mainly parallelizes along the batch and head dimensions — each (batch, head) combination gets one CTA. This utilization is very low in the batch=1 inference scenario, because batch=1 means only num_heads CTAs, far from enough to fill the 108 SMs of an A100. FA-2 also splits the sequence dimension for parallelism: different Q blocks (the outer loop index i) go to different CTAs. This way, even at batch=1 and seq_len=8192, enough CTAs are produced to fill the SMs. The cost is that each CTA has to complete the entire K/V loop independently and cannot share K/V loads — but attention itself processes all keys for each query independently, so there is no extra computation.


### 3.2 FlashAttention-3: Three Hopper-Specific Features

FA-3 (Shah et al. 2024) is a Hopper-specific implementation; its gains over FA-2 come from exploiting three new hardware features.

**Asynchronous softmax (WGMMA + softmax overlap)**. The inner loop of FA-2 is serial: first compute $QK^T$, then softmax, then $PV$. The three steps use different hardware resources — matmul uses tensor cores, softmax uses CUDA cores. FA-3 exploits the asynchronous nature of WGMMA to let the next tile's matmul overlap with the current tile's softmax inside the same warp group. Concretely, it splits each warp's execution into three segments, "issue WGMMA -> wait for WGMMA to finish -> compute softmax", and uses ping-pong scheduling to alternate between two register sets so that one warp is always computing.

**Ping-pong scheduling**. This is the key trick behind FA-3's performance gain. Hopper's WGMMA is asynchronous — after issuing the instruction, the warp can do something else and come back later to check the result. FA-3 uses two accumulator buffers (A and B). When computing tile $i$: issue tile $i$'s WGMMA -> immediately process tile $i-1$'s softmax (using buffer A) -> check tile $i$'s WGMMA completion -> issue tile $i+1$'s WGMMA -> process tile $i$'s softmax (using buffer B) -> ... . This A/B alternation lets matmul and softmax overlap almost completely, hiding the non-matmul overhead down to near zero.

**FP8 tensor core utilization**. FA-3's forward can run $QK^T$ and $PV$ on FP8 (E4M3) tensor cores, doubling the throughput of BF16. But the softmax part still accumulates in FP32 for numerical stability — the $m$, $d$, $o$ of the online softmax are all FP32, and the scale factor is per-head. This "FP8 matmul + FP32 accumulate" mixed strategy is in the same vein as DeepGEMM's per-block scaling.


### 3.3 FlashAttention Source Reading Route

The FlashAttention repository is an order of magnitude larger than DeepGEMM; the suggested entry order is below. All paths are relative to the root of the [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) repository.

**Python entry:**

- `flash_attn/flash_attn_func.py` — public API, a single attention call
- `flash_attn/flash_attn_varlen_func.py` — variable-length version, used by MoE and packing scenarios

**CUDA implementation (FA-2 + FA-3):**

- `csrc/flash_attn/flash_api.cpp` — PyTorch C++ extension entry, argument validation and launch
- `csrc/flash_attn/flash_fwd_kernel.h` — core template for the forward kernel, with separate causal / non-causal versions
- `csrc/flash_attn/flash_fwd_launch.h` — launch configuration, picks grid/block by (seqlen, headdim, dtype)
- `csrc/flash_attn/kernels_utils.h` — device functions for softmax, rescale, and scale
- `csrc/flash_attn epilogue/epilogue_fwd.hpp` — the $o / d$ normalization in the forward epilogue

**FA-3 specific (Hopper, in the `hopper` subdirectory):**

- `csrc/flash_attn/hopper/flash_fwd_kernel.h` — the FA-3 main kernel, including ping-pong scheduling
- `csrc/flash_attn/hopper/epilogue_fwd.hpp` — the FA-3 epilogue, handling FP8 scale
- `csrc/flash_attn/hopper/utils.h` — TMA descriptor construction and mbarrier synchronization

**Triton implementation (teaching-friendly):**

- `flash_attn/flash_attn_triton.py` — the complete Triton version of FlashAttention, readable line by line

Suggested route: first read `flash_attn_triton.py` to build intuition for the tiling loops (about 300 lines of Python), then cross-reference `csrc/flash_attn/flash_fwd_kernel.h` for the engineering optimizations. Look at the FA-3 part separately in the `hopper/` subdirectory — that is where the ping-pong scheduling and WGMMA overlap are implemented.


In [ ]:
# FlashAttention source reading route
fa_reading_path = [
    ("Python entry", [
        ("flash_attn/flash_attn_func.py",      "Public API for a single attention call"),
        ("flash_attn/flash_attn_varlen_func.py", "Variable-length version, used by MoE / packing"),
    ]),
    ("CUDA implementation (shared by FA-2 and FA-3)", [
        ("csrc/flash_attn/flash_api.cpp",           "PyTorch ext entry, argument validation + launch"),
        ("csrc/flash_attn/flash_fwd_kernel.h",      "forward kernel template"),
        ("csrc/flash_attn/flash_fwd_launch.h",      "launch config that picks grid/block by shape"),
        ("csrc/flash_attn/kernels_utils.h",         "device functions for softmax / rescale"),
    ]),
    ("FA-3 Hopper specific", [
        ("csrc/flash_attn/hopper/flash_fwd_kernel.h", "main kernel with ping-pong scheduling"),
        ("csrc/flash_attn/hopper/epilogue_fwd.hpp",   "epilogue that handles FP8 scale"),
        ("csrc/flash_attn/hopper/utils.h",            "TMA descriptor + mbarrier synchronization"),
    ]),
    ("Triton implementation (teaching-friendly)", [
        ("flash_attn/flash_attn_triton.py", "complete Triton version of FA, about 300 lines"),
    ]),
]

print("FlashAttention source reading route")
print("=" * 70)
for layer, files in fa_reading_path:
    print(f"\n[{layer}]")
    for path, desc in files:
        print(f"  {path}")
        print(f"      -> {desc}")
print()
print("Key observation: read the Triton version first to build intuition, then cross-reference the CUDA version for engineering optimizations.")
print("        The hopper/ subdirectory of FA-3 is the best material for learning ping-pong scheduling.")


## 4. Triton Basics: Reading block-level Programming

Triton is a GPU kernel DSL maintained by OpenAI. Its syntax resembles Python, but it generates PTX that is just as efficient as CUDA. Its core abstraction is **block-level programming** — the programmer writes "the logic for one block", and the Triton compiler is responsible for mapping that logic onto the warps and lanes of an SM.

This section does not require the reader to write Triton. The goal is to be able to read someone else's Triton kernel — for example the Triton implementation of FA, custom ops in vLLM, or fused kernels in xformers.


### 4.1 Three Core Ops: tl.load / tl.store / tl.dot

A Triton program has the following structure: a function decorated with `@triton.jit`, whose arguments include a group of "pointers" and a group of "shapes"; in the function body, `tl.load` reads data from HBM into block registers, `tl.dot` does a block-level matrix multiply, and `tl.store` writes the result back to HBM.

**`tl.load(ptr, mask, other)`**. `ptr` is a pointer to a block (usually `ptr + offset`, where offset is a `[BLOCK_SIZE]` arange); `mask` is a `[BLOCK_SIZE]` boolean tensor used to handle boundaries (for example, when the end of the sequence does not fill a whole block); `other` is the fill value when mask is False (commonly 0 or -inf). At the lower level this instruction expands into a set of global memory loads + shared memory writes, essentially `cudaMemcpyAsync` in CUDA.

**`tl.dot(a, b)`**. `a` is `[M, K]`, `b` is `[K, N]`, returns `[M, N]`. This is the only Triton op that touches the tensor core — all other operations (add, multiply, exp) use CUDA cores. This is why FlashAttention's implementation rewrites the algorithm into a "mostly matmul" form: only by minimizing the work in rescale and softmax can the tensor core stay fully fed.

**`tl.store(ptr, value, mask)`**. Symmetric to `tl.load`, writes block register values back to HBM. `mask` again handles boundaries.

Beyond these three, there are element-wise ops like `tl.arange`, `tl.exp`, `tl.maximum` whose semantics resemble PyTorch, except they operate on block registers (a `[BLOCK_SIZE]` SIMD vector), not on individual scalars.


### 4.2 A Minimal Triton Matmul

Below is the minimal matmul from the official Triton tutorial (simplified by dropping autotune and masks). It demonstrates every core pattern of block-level programming: the outer grid loop, the inner K-tiling loop, and the load/dot/store trio. The whole thing is under 30 lines.

```python
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,   # HBM base addresses of the three tensors
    M, N, K,               # shape of the matrices (scalars)
    stride_am, stride_ak,  # row / column stride of A
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, # block size (compile-time constant)
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    # each CTA is responsible for one output tile C[pid_m, pid_n]
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # compute the row / column indices inside this block
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    # initial pointers for A and B (start along the K dimension)
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    # accumulator, FP32
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # K-dimension tiling loop
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs)         # [BLOCK_M, BLOCK_K]
        b = tl.load(b_ptrs)         # [BLOCK_K, BLOCK_N]
        acc = tl.dot(a, b, acc)     # tensor core matmul
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # write back to C
    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(c_ptrs, acc)
```

Reading this code line by line reveals every core pattern of Triton:

- `tl.program_id(0)` and `tl.program_id(1)` are this CTA's coordinates in the grid, equivalent to `blockIdx.x` / `blockIdx.y` in CUDA
- `tl.arange(0, BLOCK_M)` generates a `[BLOCK_M]` arange that broadcasts against `offs_m` to form the `[BLOCK_M, BLOCK_K]` index matrix
- `a_ptrs` is not the data itself, but the pointer matrix telling the next `tl.load` where to read from
- The block sizes marked with `tl.constexpr` are compile-time constants; Triton generates a specialized PTX for every (BLOCK_M, BLOCK_N, BLOCK_K) combination
- `tl.dot(a, b, acc)` is the accumulating form of matmul, equivalent to `acc += a @ b`, but it directly invokes a tensor core instruction

Reading this matmul side by side with the PyTorch FlashAttention implementation in Appendix G, you will see the structures are almost identical — both are "outer Q-tile loop, inner K/V-tile loop, each tile does load -> compute -> update running variable". Triton's role is to translate this block-level logic written in Python into efficient PTX, so algorithm engineers do not have to write CUDA directly.


## 5. MoE-Specific Kernels: grouped GEMM and scatter/gather

In the forward path of a MoE model, the attention part is the same as in a dense model and can use standard FlashAttention. The real trouble is the FFN part: each token is routed to $k$ experts, each expert is an independent FFN, and it has to do one GEMM on the subset of tokens assigned to it. Naively looping over the $E$ experts and calling cuBLAS GEMM once per expert performs poorly — because each expert usually has very few tokens (64-512), and the compute utilization of a small-matrix GEMM is below 10%.

Grouped GEMM is the dedicated kernel designed for this scenario.


### 5.1 grouped GEMM: One Launch Handles All Experts

The input to grouped GEMM is $E$ pairs of $(A_e, B_e)$, and the output is $E$ matrices $C_e = A_e B_e$. A single kernel launch computes all experts. The key design is to treat the $E$ independent GEMMs as one "virtual large GEMM" and add one more dimension, E, to the CTA grid alongside (M, N).

There are two implementation variants. The first is **padding mode**: the token counts of all experts are padded to the same value `max_tokens`, and the grid becomes `(E, max_tokens / BLOCK_M, N / BLOCK_N)`. Each CTA knows which expert, which M tile, and which N tile it is on, and reads from the corresponding `A[e]` and `B[e]`. This implementation is simple but wasteful — the padded portion does useless work. Section 9.2 of Appendix G computes this waste in detail.

The second is **contiguous / varlen mode**: all valid tokens of all experts are concatenated into one long 1D tensor, accompanied by a `cu_seqlens` array that records the start and end of each expert. The CTA grid is `(total_tokens / BLOCK_M, N / BLOCK_N)`, and each CTA does a binary search over `cu_seqlens` to find which expert it belongs to. This implementation has no padding waste and is the standard approach in DeepGEMM and FlashAttention varlen.

DeepGEMM's grouped GEMM implementation is in `csrc/includes/kernels/grouped_gemm/kernel.cuh`, and shares the same warp specialization and epilogue infrastructure as its dense GEMM. The differences are only: the grid has an extra expert dimension, the TMA descriptor has to be constructed per expert, and the scale factor is stored per expert.


### 5.2 Triton Code Snippet for grouped GEMM

Below is a simplified Triton implementation of grouped GEMM (adapted from the style of the official Triton tutorial `09-persistent-matmul`). Compared with the normal matmul in 4.2, it adds two things: per-expert pointer offsets, and a binary search over `cu_seqlens`.

```python
@triton.jit
def grouped_matmul_kernel(
    a_ptr, b_ptr, c_ptr,      # 1D tensor after concatenating all experts
    cu_seqlens_ptr,           # [E+1], cumulative start/end of each expert
    N, K,                     # weight shape of each expert (shared)
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    EXPERTS: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)  # block index along the M dimension (global)
    pid_n = tl.program_id(1)  # block index along the N dimension

    # binary search to find which expert pid_m falls into
    # cu_seqlens = [0, e0_tokens, e0_tokens + e1_tokens, ...]
    expert_id = 0
    for e in range(EXPERTS):
        end = tl.load(cu_seqlens_ptr + e + 1)
        # if pid_m * BLOCK_M < end, it falls in this expert
        # Triton control flow requires a mask here
        expert_id = tl.where(pid_m * BLOCK_M < end, e, expert_id)

    # compute the M offset of the current block inside its expert
    start = tl.load(cu_seqlens_ptr + expert_id)
    local_m = pid_m * BLOCK_M - start

    # the rest is exactly the same as a normal matmul,
    # only the pointers have to be offset by the expert's base
    a_ptrs = a_ptr + (local_m[:, None] * stride_am +
                     tl.arange(0, BLOCK_K)[None, :] * stride_ak)
    b_ptrs = b_ptr + (expert_id * N * K +   # offset of this expert's B in the big B tensor
                      tl.arange(0, BLOCK_K)[:, None] * stride_bk +
                      tl.arange(0, BLOCK_N)[None, :] * stride_bn)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs)
        b = tl.load(b_ptrs)
        acc = tl.dot(a, b, acc)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = c_ptr + (local_m[:, None] * stride_cm +
                      tl.arange(0, BLOCK_N)[None, :] * stride_cn)
    tl.store(c_ptrs, acc)
```

Compared with the normal matmul in 4.2, the grouped version adds very little code — the core is the `expert_id` lookup at the top. This also explains why grouped GEMM is faster than looping over experts: the overhead of one kernel launch is amortized across the $E$ experts, and the GEMM logic inside each expert is identical to the dense version, with no extra complexity needed for "variable length".


### 5.3 The scatter/gather Kernel for Expert Routing

Beyond grouped GEMM, MoE has another class of dedicated kernels: scatter / gather. They handle the movement of tokens between their "original sequence position" and their "contiguous position inside an expert".

The forward process has four steps:

1. **router computation**: compute an $E$-dim logit per token, pick the top-k experts
2. **scatter** (dispatch): copy each token to the $k$ experts it is routed to; inside each expert the tokens are stored contiguously
3. **grouped GEMM**: all experts compute the FFN in parallel
4. **gather** (combine): weight and merge each expert's output back to the original sequence position by the routing weights

Scatter and gather are essentially masked memory copies — each token is written to 1-2 non-contiguous target positions (inside experts), and at read time it is read back from multiple experts and weighted. Writing this kind of kernel in CUDA requires handling many boundary conditions (variable length, top-2, expert imbalance), but in Triton it can be very concise: the core is a single `tl.load` plus a single `tl.store`, with a mask from the routing table.

Both vLLM and SGLang have this kind of scatter/gather kernel in their MoE implementations, usually fused with grouped GEMM into one fused MoE kernel — dispatch -> GEMM1 -> activation -> GEMM2 -> combine, all in one kernel launch. This fusion keeps the intermediate tensors (the output of each GEMM) entirely in SRAM, avoiding writeback to HBM, and is key to MoE inference performance.


### 5.4 Why MoE Kernels Are Harder to Write Than dense GEMM

Turning a dense GEMM into a grouped GEMM looks like just adding an expert dimension, but engineering-wise it introduces three new difficulties.

**First, variable length.** The token count per expert is not fixed — driven by the router's dynamic decisions, one expert may receive 10 tokens while another receives 1000. A grouped GEMM must support variable-length input, which means the kernel has to do boundary checks internally, store scale factors per expert, and construct TMA descriptors per expert. In a dense GEMM all of these are constants.

**Second, load imbalance.** If the router sends 90% of tokens to one expert, that expert's GEMM becomes the bottleneck while the SMs of the other experts sit idle. This cannot be solved at the kernel level; it requires the router to be trained with an auxiliary loss for load balancing (covered in Section 10). But what the kernel can do is dynamically adjust the grid at launch time, giving more CTAs to experts with more tokens. This kind of dynamic grid is unnecessary for dense GEMM.

**Third, fusion complexity.** The FFN of a dense model is three independent ops, `Linear -> activation -> Linear`, with tensors written back to HBM between them. The MoE FFN fuses these three steps into one kernel; the output size is variable length and the activation has to be done in SRAM — which means the kernel simultaneously manages the lifetime of three classes of tensors: the input scatter, the intermediate results of the two GEMMs, and the output gather. The CUTLASS Hopper MoE example (`examples/55_hopper_moe`) has a complete implementation and is the best reference for learning fused MoE kernels.


In [ ]:
# Use NumPy to simulate the data flow of MoE scatter,
# to help understand where the input of grouped GEMM comes from
import numpy as np

np.random.seed(42)

# scenario: 8 tokens, 3 experts, top-1 routing
n_tokens = 8
n_experts = 3
d = 4

# which expert each token is routed to (output of the router)
routing = np.random.randint(0, n_experts, size=n_tokens)
print(f"routing decision: {routing}")
print(f"hidden state shape per token: [{n_tokens}, {d}]")

# scatter: group by expert, tokens of each expert are stored contiguously
cu_seqlens = [0]
for e in range(n_experts):
    cu_seqlens.append(cu_seqlens[-1] + (routing == e).sum())
print(f"\ncu_seqlens (cumulative start/end): {cu_seqlens}")
print("meaning: expert 0 occupies [0, 3), expert 1 occupies [3, 6), expert 2 occupies [6, 8)")

# build the contiguous buffer after scatter
scattered = np.zeros((n_tokens, d))
perm = []
for e in range(n_experts):
    for i, r in enumerate(routing):
        if r == e:
            perm.append(i)
perm = np.array(perm)
print(f"\npermute order (original token position -> new position): {perm}")
print("meaning: move original tokens [2, 5, 7] to expert 0's positions [0, 1, 2]")
print("         because the router routed these three tokens to expert 0")

print()
print("Key observations:")
print("  the scattered buffer is contiguous and can be fed directly to a grouped GEMM kernel")
print("  cu_seqlens tells the kernel the start and end of each expert")
print("  gather is the inverse of scatter — merge back to the original positions by the routing weights")


## 6. Learning Path

For readers who want to go deeper into GPU kernel engineering, the suggested order is below. The core principle is: first build block-level intuition, then read production kernels, and only then start writing your own. Jumping straight to the last step will get you stuck in CUDA details.

**Step 1: Build hardware intuition.** Read Appendix A (GPU hardware) to understand SMs, tensor cores, and the HBM/SRAM hierarchy. Focus on the roofline model — whether an op is compute-bound or memory-bound determines the direction of optimization. No code is needed in this step.

**Step 2: Learn Triton.** Start from the official Triton tutorial `01-vector-add` and read through `06-fused-attention` in order. The first five tutorials cover all the basic ops (load / store / dot / reduce); the sixth is a simplified version of FlashAttention, and if you can read it you can see what Appendix G's PyTorch version looks like on a GPU.

**Step 3: Read the FlashAttention source.** First read the Triton version in `flash_attn/flash_attn_triton.py` (about 300 lines), then cross-reference the FA-2 CUDA version (`csrc/flash_attn/flash_fwd_kernel.h`). The goal is to understand what the two FA-2 improvements over FA-1 look like in code.

**Step 4: Read the DeepGEMM source.** Follow the route in Section 2.4 of this appendix. DeepGEMM is much smaller than FlashAttention in code size, but it involves more modern Hopper features (TMA, WGMMA, warp specialization). If you can read FA's CUDA version, you can read DeepGEMM.

**Step 5: Read the CUTLASS Hopper examples.** CUTLASS is NVIDIA's CUDA template library; all modern GEMM kernels (including DeepGEMM) are built on top of its abstractions. `examples/55_hopper_moe` is a complete MoE example, and `examples/48_hopper_fmha` is CUTLASS's implementation of FlashAttention. Reading these two examples helps you understand CUTLASS's producer/consumer model.

**Step 6: Start writing.** Begin by modifying FlashAttention's Triton version — add a causal mask, change a block size, add a head_dim branch. Change only one thing at a time, and run a benchmark to see the performance change. After you can make these modifications independently, try writing a brand new kernel.


### 6.1 Recommended Resources

**Talks / videos:**

- Phil Tillet, [Triton: An Intermediate Language and Compiler for Tiled Neural Network Computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), MAPL 2019 — the original Triton paper, explains the design motivation behind block-level programming
- Tri Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691) — multiple versions of the author's own talk are on YouTube; focus on the work partitioning part
- Tri Dao & Brian Karrer, [FlashAttention-3 talk at GPU Mode](https://www.youtube.com/watch?v=iVCQuQmBx7Y) — Hopper optimization details of FA-3, including ping-pong scheduling
- [GPU Mode Discord](https://discord.gg/gpumode) — community of GPU kernel engineers; kernel engineers from NVIDIA / OpenAI / Anthropic are all in there, with regular talks

**Paper reading order:**

1. Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022 — the original paper on online softmax + tiling
2. Dao, [FlashAttention-2](https://arxiv.org/abs/2307.08691), 2023 — work partitioning and parallelization improvements
3. Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024 — Hopper's WGMMA / TMA / FP8
4. DeepSeek-AI, [DeepGEMM Technical Report](https://github.com/deepseek-ai/DeepGEMM/blob/main/README.md) — engineering implementation of per-block scaling and BF16 accumulation
5. Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018 — the mathematical foundation of online softmax

**Code repositories:**

- [deepseek-ai/DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) — FP8 GEMM library
- [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) — the FlashAttention family
- [openai/triton](https://github.com/openai/triton) — the Triton compiler and tutorials
- [NVIDIA/cutlass](https://github.com/NVIDIA/cutlass) — CUDA template library, the infrastructure for Hopper kernels
- [triton-lang/triton](https://triton-lang.org/main/getting-started/tutorials/index.html) — official Triton tutorials

**Community and benchmarks:**

- The `#flash-attention`, `#triton`, `#cutlass` channels of [GPU Mode Discord](https://discord.gg/gpumode)
- [Stanford CRFM benchmark suite](https://crfm.stanford.edu/) — measured throughput comparisons of various kernels


## Summary

- [ ] The core constraints of advanced kernels are: the asynchronous nature of hardware instructions (TMA / WGMMA), the SRAM capacity ceiling, and the low dynamic range of FP8
- [ ] DeepGEMM uses per-block scaling to squeeze FP8 data into the E4M3 range, with the scale factor layout aligned to the TMA descriptor
- [ ] BF16 output with an FP32 accumulator is the standard for FP8 GEMM; the output stage uses stochastically rounded truncate to avoid bias
- [ ] Warp specialization splits the warps inside a CTA into producer (TMA load) and consumer (WGMMA), synchronized with mbarrier
- [ ] The two FA-2 improvements: fewer non-matmul FLOPs, and parallelism along the seq dimension so that even batch=1 can fill the SMs
- [ ] FA-3 uses ping-pong scheduling to overlap softmax and WGMMA, plus FP8 tensor cores, reaching 75% of H100 peak compute
- [ ] The three core ops of Triton are `tl.load` / `tl.store` / `tl.dot`, where `tl.dot` is the only op that touches the tensor core
- [ ] Block-level programming lets algorithm engineers write block-level logic; the Triton compiler maps it onto warps and lanes on the SM
- [ ] Grouped GEMM packs the GEMMs of E experts into a single kernel launch and uses `cu_seqlens` to handle variable length
- [ ] MoE kernels are hard to write for three reasons: variable length, load imbalance, and fusion complexity (scatter -> GEMM -> activation -> GEMM -> gather)


## Exercises

The three exercises below help you get familiar with the source structure of DeepGEMM and FlashAttention. You can ask an AI for ideas, break the problem into steps, or check your direction, but it is not recommended to let the AI "finish the problem for you" — what matters is opening the repository yourself and reading along the file paths.

**Exercise 1: Identify the responsibilities of DeepGEMM files**

Below are three file paths from the DeepGEMM repository. Match each file with its responsibility description.

Hint: Refer to the source reading route in Section 2.4.


In [ ]:
# Exercise 1: matching DeepGEMM file responsibilities
files = [
    "csrc/includes/kernels/gemm/kernel.cuh",
    "csrc/includes/kernels/gemm/epilogue.cuh",
    "csrc/includes/kernels/gemm/warp_specializer.cuh",
]
descriptions = [
    "defines the producer / consumer warp split, initializes and waits on mbarrier",
    "the main kernel template, containing the outer K loop and the core load / dot / store logic",
    "converts the FP32 accumulator to BF16 output, and applies the per-block scale factor",
]

# TODO: reorder descriptions so they correspond one-to-one with files
# Hint: kernel.cuh is the main loop, epilogue is the output stage, warp_specializer is the warp split
answer = None  # TODO: fill in a list whose elements are a permutation of the description indices, e.g. [1, 2, 0]

assert answer is not None, "please fill in the answer first"
assert len(answer) == 3, "the answer should have 3 elements"
assert sorted(answer) == [0, 1, 2], "the answer should be a permutation of [0, 1, 2]"

# correct matching:
#   kernel.cuh          -> main loop template            -> descriptions[1]
#   epilogue.cuh        -> FP32 -> BF16 + scale          -> descriptions[2]
#   warp_specializer.cuh -> producer/consumer + mbarrier  -> descriptions[0]
expected = [1, 2, 0]
assert answer == expected, (
    "Hint: kernel.cuh is the main loop (with the K tiling loop),"
    "epilogue is the output stage (FP32 -> BF16 + scale),"
    "warp_specializer defines the producer/consumer mbarrier synchronization"
)

print("Exercise 1 passed:")
for f, i in zip(files, answer):
    print(f"  {f}")
    print(f"    -> {descriptions[i]}")
print()
print("Learned: the DeepGEMM kernel / epilogue / warp_specializer trio")
print("        corresponds to the main loop, the output, and warp synchronization — this is the standard layering of a Hopper GEMM kernel.")


**Exercise 2: Identify the key file of FlashAttention-3**

The Hopper-specific code of FlashAttention-3 lives under the `csrc/flash_attn/hopper/` subdirectory. Three files are listed below; pick the one that **contains the ping-pong scheduling main kernel**.

Hint: Refer to Section 3.3. Ping-pong scheduling is the core trick behind FA-3's performance gain, letting softmax overlap with WGMMA.


In [ ]:
# Exercise 2: pick the FA-3 ping-pong scheduling main kernel file
candidates = {
    "A": "csrc/flash_attn/hopper/utils.h",
    "B": "csrc/flash_attn/hopper/flash_fwd_kernel.h",
    "C": "csrc/flash_attn/hopper/epilogue_fwd.hpp",
}

print("Candidate files:")
for k, v in candidates.items():
    print(f"  ({k}) {v}")
print()

answer = None  # TODO: fill in "A" / "B" / "C"

assert answer in {"A", "B", "C"}, "the answer must be one of A / B / C"
assert answer == "B", (
    "Hint: ping-pong scheduling is the core logic of the forward kernel,"
    "        so it lives in the _kernel_ file, not in utils or epilogue"
)

print(f"Exercise 2 passed: the FA-3 ping-pong scheduling main kernel is")
print(f"  {candidates[answer]}")
print()
print("Learned: the hopper/ subdirectory of FA-3 is layered as _kernel / epilogue / utils,")
print("        and core algorithmic logic like ping-pong scheduling always lives in the _kernel file.")


**Exercise 3: Identify the key design of grouped GEMM**

Of the three statements below, which one accurately describes the core difference of grouped GEMM relative to dense GEMM?

Hint: Refer to Section 5.1. The core of grouped GEMM is "packing E independent GEMMs into a single launch".


In [ ]:
# Exercise 3: pick the core design of grouped GEMM
options = {
    "A": (
        "grouped GEMM compresses each expert's weights to INT4,"
        "trading lower precision for higher throughput"
    ),
    "B": (
        "grouped GEMM packs the GEMMs of E experts into a single kernel launch,"
        "the grid has an extra expert dimension, and cu_seqlens handles variable length"
    ),
    "C": (
        "grouped GEMM concatenates all experts' weights into one large matrix,"
        "and calls one ordinary cuBLAS GEMM to finish the computation"
    ),
}

print("Candidate descriptions:")
for k, v in options.items():
    print(f"  ({k}) {v}")
print()

answer = None  # TODO: fill in "A" / "B" / "C"

assert answer in {"A", "B", "C"}, "the answer must be one of A / B / C"
assert answer == "B", (
    "Hint: A is quantization (unrelated to grouped), C is concatenation (still one dense GEMM, cannot handle variable-length experts);"
    "        the core of grouped GEMM is 'E GEMMs in one launch' + 'cu_seqlens for variable length'"
)

print(f"Exercise 3 passed: the core design of grouped GEMM is")
print(f"  {options[answer]}")
print()
print("Learned: grouped GEMM solves the batched computation of 'E variable-length small GEMMs',")
print("        not quantization, and not simple concatenation — it needs a specialized kernel that supports variable length.")
print("        DeepGEMM, CUTLASS, and FlashAttention varlen all implement this pattern.")


## References

- DeepSeek-AI, [DeepGEMM repository](https://github.com/deepseek-ai/DeepGEMM), 2025
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022
- Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691), 2023
- Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024
- Tillet et al., [Triton: an intermediate language and compiler for tiled neural network computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), MAPL 2019
- Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018
- NVIDIA, [CUTLASS Hopper examples](https://github.com/NVIDIA/cutlass/tree/main/examples), 2024
- [OpenAI Triton tutorials](https://triton-lang.org/main/getting-started/tutorials/index.html)
